# Session 12: Database Integration

**Course:** Python for Data Engineering  
**Phase 3:** Data Engineering Concepts

**What we'll cover:**
- SQL basics for data engineers
- Python DB connectors (sqlite3)
- CRUD operations from Python
- Using pandas with databases
- Building a CSV-to-database pipeline

**Note:** This session is demo-heavy. Follow along by running each cell. We use SQLite (built into Python, no setup needed).

In [ ]:
import sqlite3
import pandas as pd
import json
import logging
from datetime import datetime

---

## 1. SQL Basics for DE

Quick refresher — you need to know these operations:

| SQL | What it does |
|-----|--------------|
| `CREATE TABLE` | Define a table schema |
| `INSERT INTO` | Add rows |
| `SELECT` | Read data |
| `UPDATE` | Modify rows |
| `DELETE` | Remove rows |
| `DROP TABLE` | Delete the table entirely |

SQLite is a file-based database — perfect for learning. No server needed, it's built into Python.

---

## 2. Connecting to a Database

In [ ]:
# Connect to SQLite (creates the file if it doesn't exist)
conn = sqlite3.connect("data/pipeline.db")
cursor = conn.cursor()

print(f"Connected to database")
print(f"SQLite version: {sqlite3.sqlite_version}")

In [ ]:
# Create a table — using our familiar sales data schema

cursor.execute("""DROP TABLE IF EXISTS sales""")

cursor.execute("""
    CREATE TABLE sales (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date TEXT NOT NULL,
        product TEXT NOT NULL,
        quantity INTEGER NOT NULL,
        unit_price REAL NOT NULL,
        total REAL NOT NULL,
        region TEXT NOT NULL
    )
""")
conn.commit()
print("Table 'sales' created")

---

## 3. Inserting Data

In [ ]:
# Insert a single row

cursor.execute("""
    INSERT INTO sales (date, product, quantity, unit_price, total, region)
    VALUES (?, ?, ?, ?, ?, ?)
""", ("2024-01-15", "Laptop", 2, 999.99, 1999.98, "north"))

conn.commit()
print("Inserted 1 row")

In [ ]:
# Insert multiple rows at once — executemany

records = [
    ("2024-01-15", "Mouse", 10, 29.99, 299.90, "south"),
    ("2024-01-16", "Keyboard", 5, 79.50, 397.50, "north"),
    ("2024-01-17", "Laptop", 1, 999.99, 999.99, "south"),
    ("2024-01-17", "Headphones", 3, 59.99, 179.97, "north"),
    ("2024-01-18", "Laptop", 1, 999.99, 999.99, "north"),
]

cursor.executemany("""
    INSERT INTO sales (date, product, quantity, unit_price, total, region)
    VALUES (?, ?, ?, ?, ?, ?)
""", records)

conn.commit()
print(f"Inserted {len(records)} rows")

> **Important:** Always use `?` placeholders, never f-strings or string formatting for SQL values. This prevents SQL injection.

---

## 4. Querying Data

In [ ]:
# Basic SELECT

cursor.execute("SELECT * FROM sales")
rows = cursor.fetchall()

print(f"Total rows: {len(rows)}")
for row in rows:
    print(f"  {row}")

In [ ]:
# Get column names from cursor description

cursor.execute("SELECT * FROM sales")
columns = [desc[0] for desc in cursor.description]
print(f"Columns: {columns}")

In [ ]:
# Filtered query with parameters

cursor.execute("SELECT product, quantity, total FROM sales WHERE region = ?", ("north",))
north_sales = cursor.fetchall()

print("North region sales:")
for row in north_sales:
    print(f"  {row[0]:12s} | qty={row[1]} | ${row[2]:,.2f}")

In [ ]:
# Aggregation in SQL

cursor.execute("""
    SELECT region, 
           COUNT(*) as order_count,
           SUM(total) as revenue,
           ROUND(AVG(total), 2) as avg_order
    FROM sales
    GROUP BY region
    ORDER BY revenue DESC
""")

print(f"{'Region':<10} {'Orders':<8} {'Revenue':>10} {'Avg':>10}")
print("-" * 40)
for row in cursor.fetchall():
    print(f"{row[0]:<10} {row[1]:<8} ${row[2]:>9,.2f} ${row[3]:>9,.2f}")

---

## 5. Pandas + Database

Pandas can read from and write to databases directly — this is the most common approach in pipelines.

In [ ]:
# Read SQL query results directly into a DataFrame

df = pd.read_sql("SELECT * FROM sales", conn)
df

In [ ]:
# Read with a filtered query

df_laptops = pd.read_sql("SELECT * FROM sales WHERE product = 'Laptop'", conn)
print(f"Laptop orders: {len(df_laptops)}")
print(f"Total laptop revenue: ${df_laptops['total'].sum():,.2f}")

In [ ]:
# Write a DataFrame to a new table — to_sql

# Load our employees CSV
employees = pd.read_csv("data/employees.csv")
employees["name"] = employees["name"].str.strip().str.title()
employees["salary"] = employees["salary"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
employees["department"] = employees["department"].str.strip().str.lower()

# Write to database
employees.to_sql("employees", conn, if_exists="replace", index=False)
print(f"Wrote {len(employees)} employees to database")

# Verify
pd.read_sql("SELECT * FROM employees", conn)

In [ ]:
# to_sql options
# if_exists="replace" — drop and recreate table
# if_exists="append"  — add rows to existing table
# if_exists="fail"    — error if table exists (default)

# Append more data
new_employees = pd.DataFrame([
    {"name": "Lisa Park", "age": 29, "salary": 72000.0, "department": "data"},
    {"name": "Tom Chen", "age": 34, "salary": 85000.0, "department": "engineering"},
])

new_employees.to_sql("employees", conn, if_exists="append", index=False)
print(f"Appended {len(new_employees)} rows")

pd.read_sql("SELECT * FROM employees", conn)

---

## 6. CSV → Database Pipeline

A common pattern: read CSV, clean with pandas, load into database.

In [ ]:
log = logging.getLogger("db_pipeline")
log.setLevel(logging.INFO)
log.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%H:%M:%S"))
log.addHandler(handler)

def csv_to_db_pipeline(csv_path, table_name, db_path="data/pipeline.db"):
    """Read CSV, clean, load to SQLite."""
    start = datetime.now()
    log.info(f"Pipeline: {csv_path} → {table_name}")
    
    # Extract
    df = pd.read_csv(csv_path)
    log.info(f"Extracted {len(df)} rows from {csv_path}")
    
    # Transform — clean string columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    log.info("Cleaned string columns")
    
    # Load
    conn = sqlite3.connect(db_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    
    duration = (datetime.now() - start).total_seconds()
    log.info(f"Loaded {len(df)} rows to table '{table_name}' in {duration:.2f}s")
    return len(df)

# Run it
csv_to_db_pipeline("data/orders_with_dates.csv", "orders")

In [ ]:
# Verify — query the loaded data

conn = sqlite3.connect("data/pipeline.db")

# Show all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

# Query orders
print("\nOrders (first 5):")
print(pd.read_sql("SELECT * FROM orders LIMIT 5", conn))

# Run an aggregation
print("\nRevenue by region:")
print(pd.read_sql("""
    SELECT region, COUNT(*) as orders, SUM(quantity * unit_price) as revenue
    FROM orders
    GROUP BY region
    ORDER BY revenue DESC
""", conn))

conn.close()

---

## 7. Cross-Table Queries

Now that we have multiple tables in the database, we can join them with SQL.

In [ ]:
conn = sqlite3.connect("data/pipeline.db")

# Join sales and employees — which departments are generating the most work?
# This is more of a demo of cross-table queries

print("Department salary summary:")
print(pd.read_sql("""
    SELECT department, 
           COUNT(*) as headcount,
           ROUND(SUM(salary), 2) as total_salary,
           ROUND(AVG(salary), 2) as avg_salary
    FROM employees
    WHERE department != 'unknown'
    GROUP BY department
    ORDER BY total_salary DESC
""", conn))

print("\nTop customers by order count:")
print(pd.read_sql("""
    SELECT customer, 
           COUNT(*) as total_orders,
           ROUND(SUM(quantity * unit_price), 2) as total_spent
    FROM orders
    GROUP BY customer
    ORDER BY total_spent DESC
""", conn))

conn.close()

---

## Lab: Build a Database ETL Pipeline

Build a complete pipeline that:

1. Read `data/transactions.json`
2. Clean and enrich:
   - Add `total_amount = quantity * unit_price`
   - Convert date to datetime
3. Load into a `transactions` table in `data/pipeline.db`
4. Run SQL queries to produce:
   - Total revenue per customer
   - Most popular product by quantity sold
   - Revenue per day
5. Save the query results to `data/db_report.json`
6. Add logging throughout

In [ ]:
import sqlite3
import pandas as pd
import json
import logging

# Your code here


---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| `sqlite3.connect()` | Connect to database (creates file if needed) |
| `cursor.execute()` | Run SQL statements |
| `?` placeholders | Always use parameterized queries, never f-strings |
| `executemany()` | Insert multiple rows efficiently |
| `pd.read_sql()` | Query results directly into DataFrame |
| `df.to_sql()` | Write DataFrame to a table (`replace`, `append`, `fail`) |
| `conn.commit()` | Save changes (required after INSERT/UPDATE/DELETE) |
| `conn.close()` | Always close when done |

**Key patterns:**
- Use `pd.read_sql()` + `df.to_sql()` for most pipeline work — much simpler than raw cursor
- Always use `?` placeholders for parameters — prevents SQL injection
- SQLite for development/testing, PostgreSQL/MySQL for production
- Same pipeline pattern: extract → transform → load, just the target changes

**Next session:** Performance Optimization — memory usage, vectorization, and chunk processing.